In [1]:
import pyspark

In [2]:
print(pyspark.__version__)

4.3.0


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("first_look")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/13 19:29:06 WARN Utils: Your hostname, DESKTOP-8DK3P4V, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/13 19:29:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 19:29:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df_raw = spark.read.csv("data/raw/yellow_tripdata_2021-01.csv.gz")

In [6]:
df_raw.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)



In [7]:
df_raw.show(3)

+--------+--------------------+--------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------+------------+--------------------+
|     _c0|                 _c1|                 _c2|            _c3|          _c4|       _c5|               _c6|         _c7|         _c8|         _c9|       _c10| _c11|   _c12|      _c13|        _c14|                _c15|        _c16|                _c17|
+--------+--------------------+--------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_date...|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surch...|total_a

In [8]:
import time

start = time.time()
df_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/raw/yellow_tripdata_2021-01.csv.gz")
)
print(f"Seconds: {time.time() - start:.1f}")
df_inferred.printSchema()

Seconds: 16.4
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [9]:
from pyspark.sql import types as T

yellow_schema = T.StructType([
    T.StructField("VendorID", T.IntegerType()),
    T.StructField("tpep_pickup_datetime", T.TimestampType()),
    T.StructField("tpep_dropoff_datetime", T.TimestampType()),
    T.StructField("passenger_count", T.IntegerType()),
    T.StructField("trip_distance", T.DoubleType()),
    T.StructField("RatecodeID", T.IntegerType()),
    T.StructField("store_and_fwd_flag", T.StringType()),
    T.StructField("PULocationID", T.IntegerType()),
    T.StructField("DOLocationID", T.IntegerType()),
    T.StructField("payment_type", T.IntegerType()),
    T.StructField("fare_amount", T.DoubleType()),
    T.StructField("extra", T.DoubleType()),
    T.StructField("mta_tax", T.DoubleType()),
    T.StructField("tip_amount", T.DoubleType()),
    T.StructField("tolls_amount", T.DoubleType()),
    T.StructField("improvement_surcharge", T.DoubleType()),
    T.StructField("total_amount", T.DoubleType()),
    T.StructField("congestion_surcharge", T.DoubleType()),
])

start = time.time()
df = (
    spark.read
    .option("header", True)
    .schema(yellow_schema)
    .csv("data/raw/yellow_tripdata_2021-01.csv.gz")
)
print(f"Seconds: {time.time() - start:.2f}")
df.printSchema()

Seconds: 0.13
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [10]:
from pyspark.sql import functions as F

start = time.time()
df_long = (
    df.filter(F.col("trip_distance") > 2)
      .select("PULocationID", "trip_distance","total_amount")
      .withColumn("amount_per_mile", F.col("total_amount") / F.col("trip_distance"))
)
print(f"Transformations took: {time.time() - start:.3f} seconds")

Transformations took: 0.354 seconds


In [11]:
df_long.explain()

== Physical Plan ==
*(1) Project [PULocationID#151, trip_distance#148, total_amount#160, (total_amount#160 / trip_distance#148) AS amount_per_mile#163]
+- *(1) Filter (isnotnull(trip_distance#148) AND (trip_distance#148 > 2.0))
   +- FileScan csv [trip_distance#148,PULocationID#151,total_amount#160] Batched: false, DataFilters: [isnotnull(trip_distance#148), (trip_distance#148 > 2.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/raw/..., PartitionFilters: [], PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,2.0)], ReadSchema: struct<trip_distance:double,PULocationID:int,total_amount:double>




In [12]:
for i in range(2):
    start = time.time()
    print(df_long.count())
    print(f"Action {i + 1} took: {time.time() - start:.1f} seconds")

566077
Action 1 took: 5.5 seconds


566077
Action 2 took: 3.3 seconds


In [13]:
print("Partitions:", df.rdd.getNumPartitions())
df_4 = df.repartition(4)
print("After repartition:", df_4.rdd.getNumPartitions())

Partitions: 1


After repartition: 4


In [14]:
start = time.time()
df_4.write.mode("overwrite").parquet("data/pq/yellow/2021/01")
print(f"Write took: {time.time() - start:.1f} seconds")

Write took: 20.7 seconds


In [15]:
df_pq = spark.read.parquet("data/pq/yellow/2021/01")
df_pq.printSchema()
print("Partitions:", df_pq.rdd.getNumPartitions())
print("Rows:", df_pq.count())

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

Partitions: 2
Rows: 1369765
